# Retake exam task II (n-step Bellman Updates)

Suppose we have a fixed MDP (Markov Decision Process) with:
* transition probabilities $P(s'|s,a)$,
* reward function $R(s,s')$ (in this task rewards do not depend on the action taken),
* discount factor $\gamma \in (0,1]$,
* and a fixed policy $\pi(a|s)$.

If we have some initial estimate $V_0^\pi(s)$ of the true expected value $V^\pi(s)$,<br>
we can perform Bellman updates to get a better estimate. The standard Bellman update is given by:

$$ Q^\pi_{k+1}(s, a) := \sum_{s'} P(s'|s,a) \left( R(s,s') + \gamma V^\pi_{k}(s') \right) $$
$$ V^\pi_{k+1}(s) := \sum_{a} \pi(a|s) Q^\pi_{k+1}(s, a) $$

Notes:
* Here we only care about the expected value according to a fixed policy $\pi$ that does not depend on our estimates of the value function.
  We're not using a greedy or $\epsilon$-greedy policy here, so we don't have a $\max$ operator.
* One often substitutes the second equation for $V_k$ into the first, to get a direct formula for $Q_{k+1}$ depending on $Q_k$.
  Here instead we'll substitute the first equation for $Q_{k+1}$ into the second to get a formula for $V_{k+1}$ depending on $V_k$.

Your task is to:
1. Implement the (one-step) Bellman update that computes $V_{k+1}$ given $V_k$.
2. Implement a two-step Bellman update that computes $V_{k+2}$ given $V_k$, doing some precomputations to minimize the amount of work needed once $V_k$ is given.

#### Partial solutions
Solutions with the following limitations will get partial credit:
* Solutions that use any `for`/`while`/recursion loops or `torch.map`/`vmap` (including in precomputations).
* Solutions that assume `rewards` is a constant Tensor, i.e. `rewards == rewards[0,0] * torch.ones((num_states, num_states))`.
* Solutions that only implement subtask 1. (the one-step Bellman update).
* Solutions that make some useful but incomplete observations about subtask 2. (you may write short comments to prove your understanding).


#### Requirements
* Please implement your solutions only in `### YOUR CODE HERE ###` blocks. You can add test code outside, but it will be removed during our testing.
* Do not add any imports.
* All dtypes are (and should be) float32, all devices are (and should be) cpu.

In [ ]:
import numpy as np
import torch.nn
from torch import Tensor

torch.random.manual_seed(1)
np.random.seed(1)

### 1. One-step Bellman Update

In [ ]:
def bellman_update(v_value: Tensor, reward: Tensor, transition_probs: Tensor, gamma: float, policy: Tensor) -> Tensor:
    """
    Args:
    - v_value: shape (num_states,),
        v_value[s] is the current estimate of the V-value for state s.
    - transition_probs: shape (num_states, num_actions, num_states),
        transition_probs[s,a,s'] is the probability of transitioning to state s' when taking action a from state s.
    - reward: shape (num_states, num_states),
        reward[s,s'] is the immediate reward for transitioning from state s to state s'.
    - gamma: the discount factor we use.
    - policy: shape (num_states, num_actions),
        policy[s,a] is the probability of taking action a from state s.

    Output:
    - new_v_value: shape (num_states,),
        new_v_value[s] is the updated estimate of the V-value for state s.
    """
    ### YOUR CODE HERE ###
    ### === ###


In [ ]:
def test1(num_states: int = 3, num_actions: int = 2, gamma: float = 1.0) -> None:
    r = 5.0
    rewards = r * torch.ones((num_states, num_states))
    transition_probs = torch.ones((num_states, num_actions, num_states)) / num_states
    policy = torch.ones((num_states, num_actions)) / num_actions

    v_value = torch.rand((num_states,))
    expected = (v_value.mean() * gamma + r) * torch.ones((num_states,))
    obtained = bellman_update(v_value, rewards, transition_probs, gamma, policy)
    assert obtained.dtype == torch.float32, f"Expected dtype torch.float32, got {obtained.dtype}"
    assert obtained.shape == (num_states,), f"Expected shape ({num_states=},), got {obtained.shape}"
    assert torch.allclose(obtained, expected), f"Test failed:\nExpected: {expected}\nGot: {obtained}"
    print("OK")

test1()
test1(gamma=0.8)

### 2. Two-step Bellman Update with precomputations.

***Instructions***
* Below in `MDP.__init__` you can do any precomputations based on $P,R,\gamma,\pi$.
* `two_step_bellman_update` should compute $V_{k+2}$ from $V_k$ using ***as few computations (floating-point operations) as possible***.<br>
  You can assume that num_states and num_actions are roughly on the same order of magnitude.
* `bellman_update` can be left unchanged, it's only there to help you test your precomputations if you want.
* Trivial, costly implementations are given. They get zero points. 

In [ ]:
class MDP:
    def __init__(self, reward: Tensor, transition_probs: Tensor, gamma: float, policy: Tensor) -> None:
        """
        Args:
        - transition_probs: shape (num_states, num_actions, num_states),
            transition_probs[s,a,s'] is the probability of transitioning to state s' when taking action a from state s.
        - reward: shape (num_states, num_states),
            reward[s,s'] is the immediate reward for transitioning from state s to state s'.
        - gamma: the discount factor we use.
        - policy: shape (num_states, num_actions); policy[s,a] is the probability of taking action a in state s.
        """
        self.reward = reward
        self.transition_probs = transition_probs
        self.gamma = gamma
        self.policy = policy

        # Precomputations.
        ### YOUR CODE HERE ###
        ### === ###

    def bellman_update(self, v_value: Tensor) -> Tensor:
        """
        Input: V_k(s), shape (num_states,).
        Output: V_{k+1}(s), shape (num_states,).
        """
        ### YOUR CODE HERE ###
        return bellman_update(v_value, self.reward, self.transition_probs, self.gamma, self.policy)
        ### === ###

    def two_step_bellman_update(self, v_value: Tensor) -> Tensor:
        """
        Input: V_k(s), shape (num_states,).
        Output: V_{k+2}(s), shape (num_states,).
        """
        ### YOUR CODE HERE ###
        return self.bellman_update(self.bellman_update(v_value))
        ### === ###

### Basic tests for 2.

In [ ]:
def test_bellman_update(num_states: int = 3, num_actions: int = 2, gamma: float = 1.0) -> None:
    r = 5.0
    rewards = r * torch.ones((num_states, num_states))
    transition_probs = torch.ones((num_states, num_actions, num_states)) / num_states
    policy = torch.ones((num_states, num_actions)) / num_actions

    mdp = MDP(rewards, transition_probs, gamma, policy)
    v_value = torch.rand((num_states,))
    expected = (v_value.mean() * gamma + r) * torch.ones((num_states,))
    obtained = mdp.bellman_update(v_value)
    assert obtained.dtype == torch.float32, f"Expected dtype torch.float32, got {obtained.dtype}"
    assert obtained.shape == (num_states,), f"Expected shape ({num_states=},), got {obtained.shape}"
    assert torch.allclose(obtained, expected), f"Test failed:\nExpected: {expected}\nGot: {obtained}"
    print("OK")


def test_bellman_update_compare(num_states: int = 3, num_actions: int = 2, gamma: float = 1.0) -> None:
    r = 5.0
    rewards = r * torch.ones((num_states, num_states))
    transition_probs = torch.rand((num_states, num_actions, num_states))
    transition_probs /= transition_probs.sum(dim=2, keepdim=True)
    policy = torch.rand((num_states, num_actions))
    policy /= policy.sum(dim=1, keepdim=True)

    mdp = MDP(rewards, transition_probs, gamma, policy)
    v_value = torch.rand((num_states,))
    expected = bellman_update(v_value, rewards, transition_probs, gamma, policy)
    obtained = mdp.bellman_update(v_value)
    assert obtained.dtype == torch.float32, f"Expected dtype torch.float32, got {obtained.dtype}"
    assert obtained.shape == (num_states,), f"Expected shape ({num_states=},), got {obtained.shape}"
    assert torch.allclose(obtained, expected), f"Test failed:\nExpected: {expected}\nGot: {obtained}"
    print("OK")

test_bellman_update()
test_bellman_update(gamma=0.8)
test_bellman_update_compare(gamma=0.8)

In [ ]:
def test_two_step_bellman_update(num_states: int = 3, num_actions: int = 2, gamma: float = 1.0) -> None:
    r = 5.0
    rewards = r * torch.ones((num_states, num_states))
    transition_probs = torch.ones((num_states, num_actions, num_states)) / num_states
    policy = torch.ones((num_states, num_actions)) / num_actions

    mdp = MDP(rewards, transition_probs, gamma, policy)
    v_value = torch.rand((num_states,))
    expected = (v_value.mean() * gamma * gamma + r * gamma + r) * torch.ones((num_states,))
    obtained = mdp.two_step_bellman_update(v_value)
    assert obtained.dtype == torch.float32, f"Expected dtype torch.float32, got {obtained.dtype}"
    assert obtained.shape == (num_states,), f"Expected shape ({num_states=},), got {obtained.shape}"
    assert torch.allclose(obtained, expected), f"Test failed:\nExpected: {expected}\nGot: {obtained}"
    print("OK")

def test_two_step_bellman_update_compare(num_states: int = 3, num_actions: int = 2, gamma: float = 1.0) -> None:
    r = 5.0
    rewards = r * torch.ones((num_states, num_states))
    transition_probs = torch.rand((num_states, num_actions, num_states))
    transition_probs /= transition_probs.sum(dim=2, keepdim=True)
    policy = torch.rand((num_states, num_actions))
    policy /= policy.sum(dim=1, keepdim=True)

    mdp = MDP(rewards, transition_probs, gamma, policy)
    v_value = torch.rand((num_states,))
    expected = mdp.bellman_update(mdp.bellman_update(v_value))
    obtained = mdp.two_step_bellman_update(v_value)
    assert obtained.dtype == torch.float32, f"Expected dtype torch.float32, got {obtained.dtype}"
    assert obtained.shape == (num_states,), f"Expected shape ({num_states=},), got {obtained.shape}"
    assert torch.allclose(obtained, expected), f"Test failed:\nExpected: {expected}\nGot: {obtained}"
    print("OK")

test_two_step_bellman_update()
test_two_step_bellman_update(gamma=0.8)
test_two_step_bellman_update_compare(gamma=0.8)
